# 04 - Architectural / parameter-isolation strategies: CWR*, AR1

These strategies isolate parameters per-class or per-task rather than (or
in addition to) regularizing / replaying:

- **CWR\*** (Copy Weight with Reinit): freezes a shared feature extractor
  and keeps a separate final-layer weight vector per class, consolidating
  into a running average.
- **AR1**: CWR\*-style architectural isolation combined with a mild
  Synaptic-Intelligence-like regularization term on the shared body.

Both were designed with convolutional feature extractors in mind (they
were introduced on CORe50), so they're the most architecture-sensitive
strategies in this project -- if you swap in a real image benchmark, you
will likely also want to swap `SimpleMLP` for a small CNN here.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import warnings; warnings.filterwarnings("ignore")

import torch
import pandas as pd
from avalanche.models import SimpleMLP
from avalanche.training import CWRStar, AR1
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, forgetting_metrics, loss_metrics

from bench_utils import make_synthetic_benchmark
from run_utils import run_strategy

BENCHMARK_CONFIG = dict(
    n_classes=10, n_experiences=5, feature_dim=64,
    n_per_class=250, class_sep=1.6, noise=1.0, seed=0,
)
benchmark = make_synthetic_benchmark(**BENCHMARK_CONFIG)

def new_model():
    return SimpleMLP(num_classes=benchmark.n_classes, input_size=benchmark.feature_dim,
                      hidden_size=64, hidden_layers=1, drop_rate=0.0)

def new_evaluator():
    return EvaluationPlugin(
        accuracy_metrics(experience=True, stream=True),
        forgetting_metrics(experience=True, stream=True),
        loss_metrics(stream=True),
        loggers=[],
    )

all_rows = []


In [2]:
# --- CWR* ---
# CWRStar needs to know the name of the final classification layer it
# should treat specially.
model = new_model()
linear_names = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
cwr_layer_name = linear_names[-1]  # module name, NOT a "<name>.weight" parameter path
print("Using CWR* layer:", cwr_layer_name)

opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = CWRStar(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                    cwr_layer_name=cwr_layer_name, train_mb_size=32, train_epochs=3,
                    eval_mb_size=128, evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "CWR*", "architectural")
all_rows += rows
print(final)


Using CWR* layer: classifier
{'strategy': 'CWR*', 'category': 'architectural', 'after_experience': 4, 'stream_acc': 0.992, 'stream_forgetting': 0.0, 'train_seconds': 0.4129054546356201}


In [3]:
# --- AR1 ---
# AR1 builds its own internal model (a CNN by default); this is the one
# strategy in this project that isn't a great fit for flat feature-vector
# data, so we note the mismatch rather than force a bad benchmark on it.
# If you swap in SplitMNIST/SplitCIFAR10 (see notebook 00), uncomment below.

# strategy = AR1(criterion=torch.nn.CrossEntropyLoss(), train_mb_size=32, train_epochs=3,
#                eval_mb_size=128, evaluator=new_evaluator(), device="cpu")
# rows, final = run_strategy(strategy, benchmark, "AR1", "architectural")
# all_rows += rows
# print(final)
print("AR1 skipped on the synthetic flat-feature benchmark -- see markdown above.")


AR1 skipped on the synthetic flat-feature benchmark -- see markdown above.


In [4]:
df = pd.DataFrame(all_rows)
df.to_csv("../results/04_architectural.csv", index=False)
df


,strategy,category,after_experience,stream_acc,stream_forgetting
0,CWR*,architectural,0,0.218,0.0
1,CWR*,architectural,1,0.406,0.0
2,CWR*,architectural,2,0.610,0.0
3,CWR*,architectural,3,0.798,0.0
4,CWR*,architectural,4,0.992,0.0
